# Dependencies

In [1]:
!pip install -q sentence-transformers qdrant-client numpy rank_bm25 pypdfium2 transformers pillow colpali-engine torchao

# Imports

In [2]:
import json, os, glob, re, shutil
from pathlib import Path
import numpy as np
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

Device: cuda
GPU: Tesla T4
VRAM: 15.6 GB


# Model Loading

In [3]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("Qwen/Qwen3-Embedding-0.6B", device=device)

print(f"Model loaded.")
print(f"  max_seq_length: {model.max_seq_length}")
print(f"  embedding dim: {model.get_sentence_embedding_dimension()}")
print(f"  tokenizer: {type(model.tokenizer).__name__}")

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

Model loaded.
  max_seq_length: 32768
  embedding dim: 1024
  tokenizer: Qwen2Tokenizer


/tmp/ipykernel_11278/4093506145.py:7: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"  embedding dim: {model.get_sentence_embedding_dimension()}")


In [4]:
import json
import re

json_paths = [
    "/content/llm_lecture_1_slide_explanations.json",
    "/content/llm_lecture_2_slide_explanations.json",
    "/content/llm_lecture_3_slide_explanations.json",
    "/content/llm_lecture_4_slide_explanations.json",
    "/content/llm_lecture_5_slide_explanations.json",
    "/content/llm_lecture_6_slide_explanations.json",
    "/content/llm_lecture_7_slide_explanations.json",
    "/content/llm_lecture_8_slide_explanations.json",
    "/content/llm_lecture_9_slide_explanations.json",
]

assert len(json_paths) == 9, (
    f"Expected 9 JSON files, found {len(json_paths)}. "
    f"Found: {json_paths}"
)

lectures = {}

for path in json_paths:
    m = re.search(r"lecture_(\d+)", path)
    lec_num = int(m.group(1))

    with open(path, "r", encoding="utf-8") as f:
        slides = json.load(f)

    lectures[lec_num] = slides
    print(f"Lecture {lec_num}: {len(slides)} slides ({path})")

total_slides = sum(len(s) for s in lectures.values())
print(f"\nTotal slides across all lectures: {total_slides}")

Lecture 1: 62 slides (/content/llm_lecture_1_slide_explanations.json)
Lecture 2: 43 slides (/content/llm_lecture_2_slide_explanations.json)
Lecture 3: 81 slides (/content/llm_lecture_3_slide_explanations.json)
Lecture 4: 105 slides (/content/llm_lecture_4_slide_explanations.json)
Lecture 5: 67 slides (/content/llm_lecture_5_slide_explanations.json)
Lecture 6: 75 slides (/content/llm_lecture_6_slide_explanations.json)
Lecture 7: 89 slides (/content/llm_lecture_7_slide_explanations.json)
Lecture 8: 75 slides (/content/llm_lecture_8_slide_explanations.json)
Lecture 9: 67 slides (/content/llm_lecture_9_slide_explanations.json)

Total slides across all lectures: 664


# Data Chunking

In [5]:
chunks = []
for lec_num in sorted(lectures.keys()):
    for slide_idx, slide in enumerate(lectures[lec_num], start=1):
        text_to_embed = (
            f"{slide['slide_title']}\n\n"
            f"{slide['slide_paragraph']}\n\n"
            f"Key concepts: {', '.join(slide['key_concepts'])}"
        )
        chunks.append({
            "id": f"L{lec_num}-S{slide_idx}",
            "text": text_to_embed,
            "metadata": {
                "lecture": lec_num,
                "slide_number": slide_idx,
                "slide_title": slide["slide_title"],
                "key_concepts": slide["key_concepts"],
                "index_text": slide["index_text"],
                "slide_paragraph": slide["slide_paragraph"],
            },
        })

print(f"Built {len(chunks)} chunks (one per slide).")
print(f"\nFirst chunk preview:")
print(f"  ID: {chunks[0]['id']}")
print(f"  Title: {chunks[0]['metadata']['slide_title']}")
print(f"  Text length: {len(chunks[4]['text'])} chars")
print(f"  Text preview: {chunks[4]['text'][:300]}...")

Built 664 chunks (one per slide).

First chunk preview:
  ID: L1-S1
  Title: Introduction to Generative AI and LLMs
  Text length: 1520 chars
  Text preview: AI Taxonomy: Generative AI and LLMs

This slide extends the previous taxonomy by adding Large Language Models as a specific type of generative AI model. The diagram keeps the same nested structure: Artificial Intelligence is the broadest area, Machine Learning is a subset of AI, Deep Learning is a s...


In [6]:
tokenizer = model.tokenizer
max_seq_len = model.max_seq_length

token_lengths = [
    len(tokenizer.encode(c["text"], add_special_tokens=True))
    for c in chunks
]

print(f"Model max_seq_length: {max_seq_len}")
print(f"Chunk token counts:")
print(f"  min:  {min(token_lengths)}")
print(f"  max:  {max(token_lengths)}")
print(f"  mean: {np.mean(token_lengths):.1f}")
print(f"  p95:  {np.percentile(token_lengths, 95):.0f}")
print(f"  p99:  {np.percentile(token_lengths, 99):.0f}")

over_limit = [(i, t) for i, t in enumerate(token_lengths) if t > max_seq_len]
if over_limit:
    print(f"\n!!  {len(over_limit)} chunks exceed max_seq_length, raising it.")
    new_max = max(token_lengths) + 32
    model.max_seq_length = new_max
    print(f"   max_seq_length raised: {max_seq_len} -> {new_max}")
else:
    print(f"\nOK All {len(chunks)} chunks fit within {max_seq_len} tokens.")

Model max_seq_length: 32768
Chunk token counts:
  min:  99
  max:  351
  mean: 185.0
  p95:  289
  p99:  311

OK All 664 chunks fit within 32768 tokens.


# Encoding

In [7]:
texts = [c["text"] for c in chunks]

embeddings = model.encode(
    texts,
    batch_size=8,
    show_progress_bar=True,
    normalize_embeddings=True,
    convert_to_numpy=True,
)

print(f"\nEmbeddings shape: {embeddings.shape}")
print(f"Embeddings dtype: {embeddings.dtype}")
print(f"Memory size: {embeddings.nbytes / 1024:.1f} KB")
print(f"First vector norm: {np.linalg.norm(embeddings[0]):.6f}  (should be ~1.0)")

Batches:   0%|          | 0/83 [00:00<?, ?it/s]


Embeddings shape: (664, 1024)
Embeddings dtype: float32
Memory size: 2656.0 KB
First vector norm: 1.001788  (should be ~1.0)


In [8]:
os.makedirs("rag_outputs", exist_ok=True)

np.save("rag_outputs/embeddings.npy", embeddings)

chunks_for_save = [
    {"id": c["id"], "text": c["text"], "metadata": c["metadata"]}
    for c in chunks
]
with open("rag_outputs/chunks_metadata.json", "w", encoding="utf-8") as f:
    json.dump(chunks_for_save, f, indent=2, ensure_ascii=False)

loaded = np.load("rag_outputs/embeddings.npy")
assert loaded.shape == embeddings.shape
assert np.allclose(loaded, embeddings)

print(f"Saved:")
print(f"  rag_outputs/embeddings.npy        - {os.path.getsize('rag_outputs/embeddings.npy')/1024:.1f} KB")
print(f"  rag_outputs/chunks_metadata.json  - {os.path.getsize('rag_outputs/chunks_metadata.json')/1024:.1f} KB")

Saved:
  rag_outputs/embeddings.npy        - 2656.1 KB
  rag_outputs/chunks_metadata.json  - 1536.5 KB


# VectorDB (Qdrant) Setup

In [22]:
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct, NamedVector

qdrant_path = "./qdrant_db"
client = QdrantClient(path=qdrant_path)

collection_name = "llm_lecture_notes"
embedding_dim = embeddings.shape[1]

if client.collection_exists(collection_name):
    client.delete_collection(collection_name)
    print(f"Deleted existing collection '{collection_name}'.")

client.create_collection(
    collection_name=collection_name,
    vectors_config={
        "text_dense": VectorParams(size=embedding_dim, distance=Distance.COSINE),
    },
)

info = client.get_collection(collection_name)
print(f"Created collection '{collection_name}'")
print(f"  vectors: {info.config.params.vectors}")
print(f"  Qdrant storage path: {qdrant_path}")

Deleted existing collection 'llm_lecture_notes'.
Created collection 'llm_lecture_notes'
  vectors: {'text_dense': VectorParams(size=1024, distance=<Distance.COSINE: 'Cosine'>, hnsw_config=None, quantization_config=None, on_disk=None, datatype=None, multivector_config=None)}
  Qdrant storage path: ./qdrant_db


In [23]:
points = []
for i, (chunk, emb) in enumerate(zip(chunks, embeddings)):
    points.append(PointStruct(
        id=i,
        vector={"text_dense": emb.tolist()},
        payload={
            "slide_id": chunk["id"],
            "lecture": chunk["metadata"]["lecture"],
            "slide_number": chunk["metadata"]["slide_number"],
            "slide_title": chunk["metadata"]["slide_title"],
            "key_concepts": chunk["metadata"]["key_concepts"],
            "index_text": chunk["metadata"]["index_text"],
            "slide_paragraph": chunk["metadata"]["slide_paragraph"],
        },
    ))

batch_size = 100
for i in range(0, len(points), batch_size):
    client.upsert(collection_name=collection_name, points=points[i:i + batch_size])

count = client.count(collection_name=collection_name, exact=True).count
print(f"Inserted {count} points into '{collection_name}'.")
assert count == len(chunks)
print("OK Count matches.")

Inserted 664 points into 'llm_lecture_notes'.
OK Count matches.


# Retrieval Test

In [24]:
QUERY_INSTRUCTION = (
    "Given a question about LLM, NLP, and generative AI lecture topics, "
    "retrieve the most relevant lecture slide content"
)

def encode_query(query_text: str) -> np.ndarray:
    formatted = f"Instruct: {QUERY_INSTRUCTION}\nQuery: {query_text}"
    return model.encode(
        [formatted],
        normalize_embeddings=True,
        convert_to_numpy=True,
    )[0]

def retrieve(query_text: str, top_k: int = 5):
    q_vec = encode_query(query_text)
    result = client.query_points(
        collection_name=collection_name,
        query=q_vec.tolist(),
        using="text_dense",
        limit=top_k,
        with_payload=True,
    )
    return result.points

test_queries = [
    "What is the chain rule of probability in language modeling?",
    "How does temperature affect LLM output randomness?",
    "What is the difference between encoder-only and decoder-only models?",
    "How does LoRA work for fine-tuning?",
    "Explain the Markov assumption in n-gram models",
    "What is RAG and how does it ground LLM responses?",
    "How does multi-agent collaboration work in LLM systems?",
]

for q in test_queries:
    print(f"\nQ: {q}")
    hits = retrieve(q, top_k=3)
    for h in hits:
        print(f"  [score={h.score:.3f}] {h.payload['slide_id']:>8}  - {h.payload['slide_title']}")


Q: What is the chain rule of probability in language modeling?
  [score=0.790]   L1-S11  - The Language Modeling Problem: Probability of Text
  [score=0.646]   L1-S23  - Neural Language Models: Logits, Softmax, and Generation
  [score=0.634]   L1-S12  - The Language Modeling Problem: Predicting the Next Token

Q: How does temperature affect LLM output randomness?
  [score=0.754]   L1-S55  - Generative Configuration: Effect of Temperature
  [score=0.674]   L1-S54  - Generative Configuration: Temperature
  [score=0.621]   L1-S46  - Generative Configuration Parameters for Inference

Q: What is the difference between encoder-only and decoder-only models?
  [score=0.721]   L1-S31  - Encoder-Only LLMs
  [score=0.715]   L1-S29  - Transformer Architecture: Encoder-Only, Encoder-Decoder, and Decoder-Only Models
  [score=0.690]   L1-S33  - Encoder-Decoder LLMs

Q: How does LoRA work for fine-tuning?
  [score=0.759]   L4-S95  - LoRA for different tasks - task A
  [score=0.752]   L4-S93  - LoRA: 

# Dense + BM25 fused with RRF Retrieval  

In [25]:
from rank_bm25 import BM25Okapi

def bm25_tokenize(text: str):
    """Lowercase, split on non-alphanumeric, drop 1-char tokens."""
    text = text.lower()
    tokens = re.findall(r"[a-z0-9][a-z0-9\-]*", text)
    return [t for t in tokens if len(t) > 1]

bm25_corpus_tokens = [bm25_tokenize(c["text"]) for c in chunks]
bm25 = BM25Okapi(bm25_corpus_tokens)

avg_len = sum(len(t) for t in bm25_corpus_tokens) / len(bm25_corpus_tokens)
print(f"BM25 index built over {len(bm25_corpus_tokens)} chunks")
print(f"Avg tokens per chunk: {avg_len:.1f}")
print(f"Sample tokens (chunk 0, first 15): {bm25_corpus_tokens[0][:15]}")

BM25 index built over 664 chunks
Avg tokens per chunk: 138.0
Sample tokens (chunk 0, first 15): ['introduction', 'to', 'generative', 'ai', 'and', 'llms', 'this', 'opening', 'slide', 'introduces', 'the', 'lecture', 'topic', 'generative', 'artificial']


In [26]:
def dense_retrieve(query_text: str, top_k: int = 20):
    """Return list of (chunk_index, score) sorted by descending score."""
    q_vec = encode_query(query_text)
    result = client.query_points(
        collection_name=collection_name,
        query=q_vec.tolist(),
        using="text_dense",
        limit=top_k,
        with_payload=False,
    )
    return [(p.id, p.score) for p in result.points]


def bm25_retrieve(query_text: str, top_k: int = 20):
    """Return list of (chunk_index, score) sorted by descending score."""
    query_tokens = bm25_tokenize(query_text)
    scores = bm25.get_scores(query_tokens)
    top_idx = np.argsort(scores)[::-1][:top_k]
    return [(int(i), float(scores[i])) for i in top_idx]


def rrf_fuse(rankings: dict, k: int = 60, top_n: int = 10):
    """
    Reciprocal Rank Fusion.
    rankings: dict[method_name -> list of (doc_id, score)]  (only the ordering matters)
    Returns list of (doc_id, rrf_score) sorted by descending rrf_score.
    """
    rrf_scores = {}
    for _method, ranked in rankings.items():
        for rank, (doc_id, _score) in enumerate(ranked, start=1):
            rrf_scores[doc_id] = rrf_scores.get(doc_id, 0.0) + 1.0 / (k + rank)
    return sorted(rrf_scores.items(), key=lambda x: -x[1])[:top_n]


def hybrid_retrieve(query_text: str, top_k: int = 5, candidate_pool: int = 20, rrf_k: int = 60):
    """Dense + BM25 fused with RRF. Pulls a wider candidate_pool from each retriever, then fuses."""
    dense_results = dense_retrieve(query_text, top_k=candidate_pool)
    bm25_results = bm25_retrieve(query_text, top_k=candidate_pool)
    fused = rrf_fuse({"dense": dense_results, "bm25": bm25_results}, k=rrf_k, top_n=top_k)
    return [
        {"chunk": chunks[doc_id], "rrf_score": rrf_score}
        for doc_id, rrf_score in fused
    ]

In [27]:
hybrid_test_queries = [
    "What is the chain rule of probability in language modeling?",
    "How does temperature affect LLM output randomness?",
    "How does LoRA work for fine-tuning?",
    "Explain the Markov assumption in n-gram models",
    "What is RAG and how does it ground LLM responses?",
    "Explain RoPE positional encoding in Llama 3",
    "What is FSDP and how does it shard model states?",
]

for q in hybrid_test_queries:
    print(f"\n{'=' * 90}")
    print(f"Q: {q}")
    print('=' * 90)

    dense_top = dense_retrieve(q, top_k=3)
    bm25_top = bm25_retrieve(q, top_k=3)
    fused_top = hybrid_retrieve(q, top_k=3, candidate_pool=20)

    print(f"\n  DENSE only:")
    for doc_id, score in dense_top:
        c = chunks[doc_id]
        print(f"    [{score:.3f}]  {c['id']:>8}  - {c['metadata']['slide_title']}")

    print(f"\n  BM25 only:")
    for doc_id, score in bm25_top:
        c = chunks[doc_id]
        print(f"    [{score:.3f}]  {c['id']:>8}  - {c['metadata']['slide_title']}")

    print(f"\n  HYBRID (RRF, k=60):")
    for r in fused_top:
        c = r['chunk']
        print(f"    [{r['rrf_score']:.4f}]  {c['id']:>8}  - {c['metadata']['slide_title']}")


Q: What is the chain rule of probability in language modeling?

  DENSE only:
    [0.790]    L1-S11  - The Language Modeling Problem: Probability of Text
    [0.646]    L1-S23  - Neural Language Models: Logits, Softmax, and Generation
    [0.634]    L1-S12  - The Language Modeling Problem: Predicting the Next Token

  BM25 only:
    [37.272]    L1-S11  - The Language Modeling Problem: Probability of Text
    [20.080]    L5-S40  - PPO Policy Loss: Model Probability Distribution over Tokens
    [18.538]    L1-S18  - Bigram Probability

  HYBRID (RRF, k=60):
    [0.0328]    L1-S11  - The Language Modeling Problem: Probability of Text
    [0.0312]    L1-S13  - How Language Models Work
    [0.0306]    L1-S18  - Bigram Probability

Q: How does temperature affect LLM output randomness?

  DENSE only:
    [0.754]    L1-S55  - Generative Configuration: Effect of Temperature
    [0.674]    L1-S54  - Generative Configuration: Temperature
    [0.621]    L1-S46  - Generative Configuration Paramete

# Answer Generation

In [15]:
import gc
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f"GPU memory before VLM load: {torch.cuda.memory_allocated()/1e9:.2f} GB used")

from transformers import AutoProcessor, AutoModelForImageTextToText

VLM_MODEL_ID = "google/gemma-4-E2B-it"
# If you hit OOM, try: "google/gemma-4-E2B-it"

vlm_processor = AutoProcessor.from_pretrained(VLM_MODEL_ID, padding_side="left")
vlm_model = AutoModelForImageTextToText.from_pretrained(
    VLM_MODEL_ID,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    attn_implementation="sdpa",
).eval()

print(f"\nLoaded VLM: {VLM_MODEL_ID}")
if torch.cuda.is_available():
    print(f"GPU memory after VLM load:  {torch.cuda.memory_allocated()/1e9:.2f} GB used")

GPU memory before VLM load: 1.20 GB used


processor_config.json: 0.00B [00:00, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/10.2G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/1951 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/208 [00:00<?, ?B/s]


Loaded VLM: google/gemma-4-E2B-it
GPU memory after VLM load:  11.41 GB used


In [30]:
SYSTEM_PROMPT = """You are a professor teaching a university course on Generative AI and Large Language Models. A student has just asked you a question during or after a lecture.

Explain your answer the way a great professor would: build intuition first, then add precision. Use analogies or concrete examples where they genuinely help. Connect ideas to the bigger picture of the course when relevant. Write in a warm, confident academic voice — clear enough for a student hearing this for the first time, rigorous enough to hold up to scrutiny.

STRUCTURE YOUR ANSWER AS FOLLOWS:
1. Open with a direct, plain-English answer to the question.
2. Build the explanation from the ground up — motivate *why* before explaining *how*.
3. If the concept has a common misconception or a subtle gotcha, call it out explicitly.
4. Close with a one-sentence takeaway that crystallises the key insight.

CITATION RULES:
- Every factual claim must be cited inline using the format [Lecture X, Slide Y].
- Multiple citations can be combined: [Lecture 1, Slide 3; Lecture 2, Slide 7].
- If the answer cannot be supported by the provided notes, say so explicitly. Do not invent information.
- Use bullet lists only when the question genuinely calls for enumeration — prefer flowing prose otherwise."""

def build_context_block(c: dict) -> str:
    """Format one retrieved chunk into a labelled, self-contained context block."""
    meta = c["metadata"]
    lines = [
        f"[Lecture {meta['lecture']}, Slide {meta['slide_number']}: \"{meta['slide_title']}\"]",
        meta["slide_paragraph"],
    ]
    if meta.get("key_concepts"):
        lines.append(f"Key concepts: {', '.join(meta['key_concepts'])}")
    if meta.get("index_text"):
        lines.append(f"Index: {meta['index_text']}")
    return "\n".join(lines)


def answer_with_citations(
    question: str,
    top_k: int = 5,
    candidate_pool: int = 20,
    max_new_tokens: int = 10000,
    show_retrieved: bool = True,
) -> dict:
    """
    Full RAG pipeline:
      1. Hybrid retrieve (dense + BM25 via RRF)
      2. Build text-only prompt from retrieved chunks
      3. Generate answer with Gemma 4
      4. Return answer string + retrieved chunks
    """

    # ── 1. Retrieve ──────────────────────────────────────────────────────────
    retrieved = hybrid_retrieve(question, top_k=top_k, candidate_pool=candidate_pool)

    if show_retrieved:
        print(f"Retrieved {len(retrieved)} chunks:")
        for r in retrieved:
            m = r["chunk"]["metadata"]
            print(f"  [{r['rrf_score']:.4f}]  L{m['lecture']}-S{m['slide_number']:02d}"
                  f"  \"{m['slide_title']}\"")
        print()

    # ── 2. Build prompt ───────────────────────────────────────────────────────
    context_blocks = "\n\n---\n\n".join(
        build_context_block(r["chunk"]) for r in retrieved
    )

    user_text = (
        f"LECTURE NOTES:\n\n"
        f"{context_blocks}\n\n"
        f"---\n\n"
        f"Student question: {question}"
    )

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": user_text},
    ]

    # ── 3. Tokenise ───────────────────────────────────────────────────────────
    inputs = vlm_processor.apply_chat_template(
        messages,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
        add_generation_prompt=True,
    ).to(vlm_model.device)

    input_len = inputs["input_ids"].shape[-1]
    print(f"Prompt tokens: {input_len}")

    # ── 4. Generate ───────────────────────────────────────────────────────────
    with torch.inference_mode():
        outputs = vlm_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
        )

    response = vlm_processor.decode(
        outputs[0][input_len:],
        skip_special_tokens=True,
    )

    return {"answer": response.strip(), "retrieved": retrieved}

In [32]:
demo_questions = [
    "What is the Markov assumption and why is it used in language modeling?",
    "Explain how LoRA reduces the cost of fine-tuning compared to full fine-tuning.",
    "What does temperature do in LLM inference?",
    "Compare encoder-only, decoder-only, and encoder-decoder transformer architectures.",
]

for q in demo_questions:
    print("=" * 90)
    print(f"Q: {q}")
    print("=" * 90)
    result = answer_with_citations(q, top_k=5, show_retrieved=True)
    print("Answer:\n")
    print(result["answer"])
    print("\n")

Q: What is the Markov assumption and why is it used in language modeling?
Retrieved 5 chunks:
  [0.0328]  L1-S14  "Markov Assumption"
  [0.0323]  L1-S13  "How Language Models Work"
  [0.0317]  L1-S15  "Markov Assumption Example"
  [0.0308]  L1-S16  "n-gram Language Models"
  [0.0299]  L1-S11  "The Language Modeling Problem: Probability of Text"

Prompt tokens: 2075
Answer:

That is an excellent and fundamental question. Understanding the Markov assumption is key to bridging the gap between simple statistical models and the complex, powerful neural networks we use today.

**In short, the Markov assumption is a simplifying heuristic that allows us to make the problem of calculating the probability of an entire sequence tractable by assuming that the next element in the sequence only depends on a fixed, limited number of immediately preceding elements, rather than the entire history.** [Lecture 1, Slide 14]

### Building the Intuition: Why We Need Simplification

To understand *why* we us